# 27 · Load tests that match reality
Level L5 · Part 4 · core · ~25 min read, ~36 min live · builds on 16, 18, 26

Time: about 25 min to read. A live run starts the pinned vLLM CPU image once and takes about 36 min. Most of that is the first request of a new server: about 10 minutes (604 s and 618 s in this lesson's two live runs, on the course machine: 4 CPU cores, no GPU). If one sitting is too long for you, Steps 7 and 8 can move to a follow-up lesson. Without Docker, the cells that start the server or send load are skipped, and Step 6 prints the **recorded output** of a live run of this notebook (`data/recorded/27.json`), labelled as such. No key is needed.

Machine: Docker, 4 CPU cores and 15 GB of RAM (the course machine; no GPU). Ports: 8270 (the lab server) and nothing else.

## Goal
By the end you can drive a server with Poisson arrivals, read p95 TTFT, p95 TPOT and goodput against an SLO, bracket the rate where the tail breaks, and tell an open-loop load tool from a closed-loop one.

## Why it matters
- **Without it:** a common first test sends the same request over and over, or as fast as the server takes them. You get one throughput number for one kind of traffic, and never meet the queues that real users cause.
- **With it:** you set an arrival rate and read the tail users feel. At 0.40 requests per second, past the lab's capacity, p95 TTFT is 21.7 s (the SLO is 5 s) and goodput is 0.068 of the 0.40 offered: most requests miss the target and wait in a queue.
- **Spreading lengths is theory here:** a long request holds a slot longer, which a fixed-length test never does. The fixed and spread runs in Step 4 look alike at 16 requests, so they prove nothing. Only hundreds of requests can show the effect.
- **When you don't need it:** a batch job with fixed work, or a smoke test that only checks the server answers.

## The idea
A load test makes two choices. The **arrival process**: an *open-loop* test sends requests on its own clock, with Poisson gaps, as real users do. A *closed-loop* test lets each user wait for an answer before asking again, so a slow server slows the test and hides the queue. The **length distribution**: prompts and answers vary, and the tail comes from the long ones.

Terms, defined once: **TTFT** is the wait before the first answer token; **TPOT** is the mean time per output token after the first; **e2e** is the time from sending a request to its last token. **Goodput** is the requests per second that meet every SLO target (here p95 TTFT under 5 s and TPOT under 0.5 s), so a fast request that misses its target does not count. (ITL, the gap between consecutive tokens, appears in GuideLLM's output in Step 7.)

## Picture
```mermaid
flowchart LR
  U[Users: random gaps, varied lengths] -->|open loop| G[Load generator]
  G --> S[Server: 4 slots]
  S --> Q[Queue when all slots are busy]
  Q --> M[p95 TTFT, TPOT and goodput]
```

## Step by step

### Step 1: the trace and the lab profile
The trace from lesson 07 is made-up support traffic (a fixed-seed generator, not a real log, so no licence to check). It is there to show that real answers vary a lot: the answers below run from 24 to 199 tokens. Its prompts are about 1,500 tokens, a **reference** for the course's shared policy prompt. The lab is a 4-core CPU, where a 1,500-token prompt took about 20 s to first token with prefix caching off (lesson 18). So the bench uses 256 prompt tokens and 48 answer tokens, each varied by plus or minus 50%. That is narrower than the trace's answer range, on purpose: this lab server is small.

In [1]:
import json, statistics
from pathlib import Path
outs = sorted(json.loads(l)["output_tokens"] for l in Path("../data/07/support_trace.jsonl").read_text().splitlines())
print(f"lesson 07 trace: {len(outs)} requests, answers {outs[0]}-{outs[-1]} tokens, median {statistics.median(outs):.0f}")

lesson 07 trace: 30 requests, answers 24-199 tokens, median 60


The lab settings: the pinned image, this lesson's container name and port, the model, and the flags of lesson 16. Targets are the lengths the bench draws around.

In [2]:
IMAGE = "vllm/vllm-openai-cpu:v0.32.0"            # pinned: the same image as lessons 16 and 18
NAME, PORT = "tut-kv-forge-15h-27", 8270          # this lesson's container and port
BASE = f"http://127.0.0.1:{PORT}"
MODEL, SERVED = "Qwen/Qwen2.5-0.5B-Instruct", "brightside-qwen-0.5b"
TARGET = [MODEL, "--served-model-name", SERVED, "--host", "0.0.0.0", "--dtype", "float32", "--enforce-eager",
          "--max-model-len", "2048", "--max-num-seqs", "4", "--max-num-batched-tokens", "2048",
          "--kv-cache-memory-bytes", str(2 * 1024**3)]       # the lesson 16 flags, prefix caching on (the default)
PROMPT_LEN, OUT_LEN, SPREAD = 256, 48, 0.5        # target lengths; SPREAD 0.5 = each length varies by plus or minus 50%
SLO_TTFT_MS, SLO_TPOT_MS = 5000, 500              # the lab SLO: p95 TTFT under 5 s, TPOT under 0.5 s (a lab choice)
HF_DIR, RES, RESULTS = Path.home() / ".cache" / "huggingface", Path("../data/27").resolve(), {}
RES.mkdir(parents=True, exist_ok=True); Path("../apps/27").mkdir(parents=True, exist_ok=True)   # the Locust file: Step 8

### Step 2: start the lab server
The server is the lesson 16 setup on this lesson's port. The helpers are the same as lessons 16 and 18: start the container, poll `/health`, and stop it at the end. The server takes no key, which is the lab setting. The first request of a new server is the slow one: the first-request cell below runs it in a background thread, so you can see how long it takes.

In [3]:
import atexit, csv, os, socket, subprocess, sys, threading, time
import httpx

def server_up():
    try: return httpx.get(f"{BASE}/health", timeout=2).status_code == 200
    except httpx.TransportError: return False

def start_server():
    """Start the pinned lab server in the background; the next cells wait for it."""
    subprocess.run(["docker", "rm", "-f", NAME], capture_output=True)
    if socket.socket().connect_ex(("127.0.0.1", PORT)) == 0:
        raise RuntimeError(f"port {PORT} is busy: stop the other process first")
    subprocess.run(["docker", "run", "-d", "--name", NAME, "-p", f"{PORT}:8000", "--shm-size", "4g",
                    "-v", f"{HF_DIR}:/root/.cache/huggingface", IMAGE, *TARGET], capture_output=True, check=True)
    atexit.register(lambda: subprocess.run(["docker", "rm", "-f", NAME], capture_output=True))

def wait_healthy(limit=900):
    """Poll /health. A cold CPU start can take several minutes."""
    deadline = time.time() + limit
    while not server_up() and time.time() < deadline:
        time.sleep(2)
    if not server_up():
        raise TimeoutError(f"server not healthy after {limit} s on port {PORT}; run this cell again")

Start the container. The call returns at once; the next cells wait for it.

In [4]:
start_server()

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


Wait for `/health`. A cold CPU start can take several minutes.

In [5]:
wait_healthy()

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


Send one short request. The first request of a new server is the slow one (about 10 minutes here), so it runs in a background thread: if the cell reports that it is still running, run the cell again.

In [6]:
def first_request(limit=1500):
    FIRST = globals().get("FIRST")
    if FIRST is None:
        FIRST = globals()["FIRST"] = {"t0": time.time(), "ok": False}
        def go(state=FIRST):
            httpx.post(f"{BASE}/v1/completions", timeout=3600, json={"model": SERVED, "prompt": "Hello",
                       "max_tokens": 4, "temperature": 0}).raise_for_status()
            state["ok"] = True
        threading.Thread(target=go, daemon=True).start()
    deadline = time.time() + limit
    while not FIRST["ok"] and time.time() < deadline:
        time.sleep(2)
    if not FIRST["ok"]:
        raise TimeoutError(f"first request still running after {time.time() - FIRST['t0']:.0f} s: run this cell again")
    print(f"first request answered after {time.time() - FIRST['t0']:.0f} s")
first_request()

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


### Step 3: the open-loop bench
`vllm bench serve` ships in the same pinned image, so the cell runs it there with `--network host`. Requests arrive with Poisson gaps at `--request-rate`, which is open loop. `--burstiness 1.0` is the Poisson setting: below 1 the arrivals clump, above 1 they spread more evenly. `--ignore-eos` makes each answer run to its sampled length, so the output length is exactly what the bench drew. The lengths come from the random dataset: each length is drawn within plus or minus `--random-range-ratio` of its target. The helper returns the numbers the table needs, read from the saved JSON.

In [7]:
def bench(tag, rate, spread, prompts=16):
    """Open loop: `prompts` requests arrive with Poisson gaps at `rate` per second; lengths vary by `spread`."""
    args = ["--endpoint", "/v1/completions", "--model", SERVED, "--tokenizer", MODEL, "--dataset-name", "random",
            "--random-input-len", str(PROMPT_LEN), "--random-output-len", str(OUT_LEN), "--random-range-ratio", str(spread),
            "--num-prompts", str(prompts), "--request-rate", str(rate), "--burstiness", "1.0", "--seed", "27",
            "--ignore-eos", "--ready-check-timeout-sec", "1200", "--percentile-metrics", "ttft,tpot,itl,e2el",
            "--metric-percentiles", "50,95", "--goodput", f"ttft:{SLO_TTFT_MS}", f"tpot:{SLO_TPOT_MS}",
            "--save-result", "--result-dir", "/results", "--result-filename", f"{tag}.json"]
    docker = ["docker", "run", "--rm", "--network", "host", "--user", f"{os.getuid()}:{os.getgid()}",
              "-e", "HOME=/tmp", "-e", "HF_HOME=/hf", "-v", f"{HF_DIR}:/hf", "-v", f"{RES}:/results",
              "--entrypoint", "vllm", IMAGE, "bench", "serve", "--backend", "openai", "--base-url", BASE, *args]
    run = subprocess.run(docker, capture_output=True, text=True, timeout=3000)
    if run.returncode != 0:
        raise RuntimeError("vllm bench failed: " + run.stderr[-400:])
    d = json.loads((RES / f"{tag}.json").read_text()); n = d["completed"]
    return {"rate": rate, "done": n, "goodput_rps": round(d["request_goodput"], 3),
            "mean_in": round(d["total_input_tokens"] / n), "mean_out": round(d["total_output_tokens"] / n),
            "p50_ttft_ms": round(d["p50_ttft_ms"]), "p95_ttft_ms": round(d["p95_ttft_ms"]),
            "p95_tpot_ms": round(d["p95_tpot_ms"]), "p95_e2el_ms": round(d["p95_e2el_ms"]), "mean_e2el_ms": round(d["mean_e2el_ms"])}

A helper that prints the rows as one table. Columns: `done` is the number of requests that completed; `goodput/s` counts only those that met the SLO.

In [8]:
def show(title, rows):
    print(title, "\n  rate/s  done  goodput/s  mean in  mean out  p50 TTFT  p95 TTFT  p95 TPOT  p95 e2e (ms)")
    for r in rows:
        print(f"  {r['rate']:6.2f}{r['done']:6}{r['goodput_rps']:>11.3f}{r['mean_in']:>9}{r['mean_out']:>10}{r['p50_ttft_ms']:>10,}{r['p95_ttft_ms']:>10,}{r['p95_tpot_ms']:>10,}{r['p95_e2el_ms']:>10,}")

### Step 4: about the same mean, fixed vs spread
The first run warms the engine with 8 requests and does not measure them. The next two runs both arrive at 0.10 requests per second, 16 requests each. They ran right after that short warm-up, so the engine was probably still cold for them: read these two rows with caution. The fixed run uses the target lengths every time. The spread run draws each length within plus or minus 50%. The two means are about the same (256/48 target, 247/46 measured).

In [9]:
bench("warmup", 0.20, SPREAD, prompts=8)          # not measured: warms the engine

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


Fixed lengths: every request asks for exactly the target lengths.

In [10]:
RESULTS["fixed_0.10"] = bench("fixed_0.10", 0.10, 0.0)

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


Spread lengths: each length is drawn within plus or minus 50% of its target.

In [11]:
RESULTS["spread_0.10"] = bench("spread_0.10", 0.10, SPREAD)

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


**Read it.** In the recorded run the two rows are within the noise of 16 requests: p95 TTFT 4.8 s fixed vs 4.8 s spread (4,815 and 4,751 ms), p95 end-to-end 24.1 s and 26.8 s. **No difference shows, and 16 requests cannot show one.** These rows also ran after only an 8-request warm-up, so they may include cold-start cost: do not read their absolute values as steady state. The reason to spread lengths is how slots work: a long request holds a slot longer, which a fixed-length test never does. That is theory here. Showing it on the tail needs hundreds of requests.

### Step 5: the rate sweep
Now the spread profile at more rates: 0.05, 0.10 (Step 4), 0.15 and 0.40 requests per second. Four slots and 12 to 18 s per request (the mean end-to-end time in the rows below the knee) give a capacity estimate of roughly 4 / 15, about 0.27 requests per second. That is an estimate, not a measurement: no rate between 0.15 and 0.40 was run. The first three rates sit under it. 0.40 is past it, so a queue builds.

0.05 requests per second: well under the capacity estimate.

In [12]:
RESULTS["spread_0.05"] = bench("spread_0.05", 0.05, SPREAD)

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


0.15 requests per second: under the capacity estimate.

In [13]:
RESULTS["spread_0.15"] = bench("spread_0.15", 0.15, SPREAD)

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


0.40 requests per second: past the capacity estimate, so the queue builds.

In [14]:
RESULTS["spread_0.40"] = bench("spread_0.40", 0.40, SPREAD)

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


**Read it.** At 0.05 and 0.15 requests per second the p95 TTFT stays under the 5 s SLO (1.6 s and 1.8 s), and goodput tracks the offered rate (0.135 at 0.15). The 0.10 row (Step 4) ran after a short warm-up and may still be cold, so we do not use it to place the knee. At 0.40 the p95 TTFT is 22 s, goodput falls to 0.07 requests per second, and about 10 requests are in the server on average against 4 slots. The extra requests wait in a queue. The knee lies between 0.15 and 0.40 in this run: bracketed, not located.

### Step 6: read the results
If Docker was not run, this cell loads the recorded run of this notebook (`data/recorded/27.json`) and says so. The table is the same either way. Steps 7 and 8 need the server, so without Docker their rows come from the recording too.

In [15]:
LIVE = bool(RESULTS)
if not LIVE:                                      # no Docker: load the recorded run
    rec = json.loads(Path("../data/recorded/27.json").read_text()); RESULTS = rec["results"]
    print("source: recorded output,", rec["note"])
show("fixed vs spread, about the same mean lengths, 0.10/s:", [RESULTS["fixed_0.10"], RESULTS["spread_0.10"]])
show("sweep, spread lengths:", [RESULTS[k] for k in sorted(RESULTS) if k.startswith("spread")])
if "guidellm" in RESULTS and "locust" in RESULTS:      # the other two tools, same server
    print(f"GuideLLM, open loop 0.10/s: {RESULTS['guidellm']['rps']} req/s, p95 TTFT {RESULTS['guidellm']['p95_ttft_ms']:,} ms, p95 ITL {RESULTS['guidellm']['p95_itl_ms']} ms")
    print(f"Locust, closed loop, {RESULTS['locust']['users']} users: {RESULTS['locust']['requests']} req, {RESULTS['locust']['rps']} req/s, p50 {RESULTS['locust']['p50_ms']:,} ms, p95 {RESULTS['locust']['p95_ms']:,} ms")

source: recorded output, a live run of this notebook on the course machine (Docker, pinned CPU image, 4 CPU cores, no GPU), 2026-10-23; the first request of the new server took 618 s; the whole run took about 36 min
fixed vs spread, about the same mean lengths, 0.10/s: 
  rate/s  done  goodput/s  mean in  mean out  p50 TTFT  p95 TTFT  p95 TPOT  p95 e2e (ms)
    0.10    16      0.084      256        48     3,225     4,815       427    24,077
    0.10    16      0.091      247        46     3,522     4,751       439    26,818
sweep, spread lengths: 
  rate/s  done  goodput/s  mean in  mean out  p50 TTFT  p95 TTFT  p95 TPOT  p95 e2e (ms)
    0.05    16      0.048      247        46     1,033     1,644       279    17,779
    0.10    16      0.091      247        46     3,522     4,751       439    26,818
    0.15    16      0.135      247        46     1,071     1,799       325    21,027
    0.40    16      0.068      247        46    11,041    21,719       374    35,136
GuideLLM, open lo

### Step 7: a second tool disagrees
GuideLLM is the vLLM project's SLO-oriented tool. Its Poisson profile sends open-loop traffic at 0.10 requests per second, with synthetic prompts and answers that vary around the same targets. Its ITL is the p95 gap between consecutive tokens. Compare its row with the vLLM bench row at the same rate: in the recorded run its p95 TTFT was about 41 s, against 4.8 s from the bench.

**Which number to use for the capacity decision: neither yet.** Both rows come from 16 requests, so the p95 is set by the two slowest requests, and the tools disagree by about 8x. We did not find the cause. The likeliest unverified suspects are the length draws, the warm state of the engine, and how each tool times its requests. Do not trust a single 16-request row: rerun both tools with hundreds of requests, check the lengths each logged, and only then size capacity.

In [16]:
out = RES / "guidellm"; out.mkdir(exist_ok=True)
spec = (f"kind=synthetic_text,prompt_tokens={PROMPT_LEN},prompt_tokens_stdev=64,prompt_tokens_min=128,prompt_tokens_max=384,"
        f"output_tokens={OUT_LEN},output_tokens_stdev=12,output_tokens_min=24,output_tokens_max=72")
subprocess.run([str(Path(sys.executable).parent / "guidellm"), "run",
    "--backend", f"kind=openai_http,target={BASE},model={SERVED}", "--profile", "kind=poisson,rate=0.1",
    "--constraint", "kind=max_requests,count=16", "--data", spec, "--tokenizer", f"kind=huggingface_auto,model={MODEL}",
    "--output", "kind=json", "--disable-console-interactive"], cwd=out, capture_output=True, text=True, check=True, timeout=1500)   # quiet; a failure stops the cell
m = json.loads((out / "benchmarks.json").read_text())["benchmarks"][0]["metrics"]
RESULTS["guidellm"] = {"rate": 0.10, "rps": round(m["requests_per_second"]["successful"]["mean"], 3),
                       "p95_ttft_ms": round(m["time_to_first_token_ms"]["successful"]["percentiles"]["p95"]),
                       "p95_itl_ms": round(m["inter_token_latency_ms"]["successful"]["percentiles"]["p95"], 1)}
print("GuideLLM at 0.10/s:", RESULTS["guidellm"])

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


### Step 8: Locust, a closed-loop test
Locust is a general-purpose load generator. Its users are closed loop: each user waits a random gap, asks one question, waits for the **whole** answer, and repeats. The arrival rate therefore depends on how fast the server answers. Locust measures the full answer, not the first token, because this request does not stream. Read its numbers as a different kind of test, not as a rival to the two open-loop tools.

Its gate: Locust sends traffic to the server above, so it needs the server running and is tagged with the server cells. It runs on your machine, not in Docker, so it needs the `locust` package in your venv.

In [17]:
%%writefile ../apps/27/locustfile.py
import random
from locust import HttpUser, task

random.seed(27)
PROMPT = "Brightside policy: " + " ".join(f"Rule {i} covers refunds, delivery and warranty claims." for i in range(1, 25))

class Desk(HttpUser):                             # one support agent: wait, ask, wait for the whole answer, repeat
    def wait_time(self):
        return random.expovariate(1 / 8)          # mean 8 s between questions
    @task
    def ask(self):
        self.client.post("/v1/completions", name="completions", json={"model": "brightside-qwen-0.5b",
            "prompt": PROMPT, "max_tokens": random.randint(24, 72), "temperature": 0})

Overwriting ../apps/27/locustfile.py


Run Locust headless: 4 simulated users for 120 seconds, then read the statistics it writes as CSV. Recorded result: 4 users, 20 requests, 0.17 requests per second, p50 16 s, p95 24 s.

In [18]:
subprocess.run([sys.executable, "-m", "locust", "-f", "../apps/27/locustfile.py", "--headless", "-u", "4", "-r", "4",
    "-t", "120s", "--host", BASE, "--csv", str(RES / "locust"), "--only-summary"], capture_output=True, text=True, timeout=900)
row = next(r for r in csv.DictReader(open(RES / "locust_stats.csv")) if r["Name"] == "completions")
RESULTS["locust"] = {"users": 4, "requests": int(row["Request Count"]), "rps": round(float(row["Requests/s"]), 3),
                     "p50_ms": int(float(row["50%"])), "p95_ms": int(float(row["95%"]))}
print("Locust, 4 users:", RESULTS["locust"])

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


Stop the container and save the live numbers, so the recording can be made from this run.

In [19]:
subprocess.run(["docker", "rm", "-f", NAME], capture_output=True)
if LIVE: (RES / "results.json").write_text(json.dumps(RESULTS, indent=1))   # the live numbers, for the recording

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


## At work
- **Databricks**' inference performance guide says to benchmark on your own deployment: throughput depends on the hardware, the batch and the lengths, so a vendor's tokens per second is a starting point, not a promise. → lesson 28 (cost per token) uses the same numbers to price a fleet.
- **Cloudflare**'s load balancer tracks the estimated in-flight prefill and decode tokens of each endpoint, so a long prompt counts for more than a short one. → lesson 30 (routing).
- **Google Cloud (GKE)**'s inference autoscaling guidance scales on queue size (throughput and cost) or batch size (tail latency), not on GPU utilisation. → lesson 31 (autoscaling on queue depth).

## Common mistakes
**1. Reading a run by a count or an average.** A row can look fine on one number and fail on the one that matters. The cell below checks two things on the row past the knee. **This cell fails on purpose**: its first check passes, its second does not.

In [20]:
r = RESULTS["spread_0.40"]                        # the row past the knee; this cell fails on purpose
assert r["done"] == 16, "all 16 requests completed"          # passes: the count looks fine
assert r["goodput_rps"] >= 0.9 * r["rate"], f"goodput {r['goodput_rps']} req/s is under 90% of the offered {r['rate']} req/s"

AssertionError: goodput 0.068 req/s is under 90% of the offered 0.4 req/s

**The fix:** read the p95 and the goodput against the offered rate, and find the highest rate whose goodput still tracks it. A completed count says nothing about the requests that missed the SLO.

## Production checklist
- Use an open-loop tool (`vllm bench serve` or GuideLLM) for capacity. Closed-loop tools hide queues.
- Spread the lengths around the real profile, use hundreds of requests, and keep a warm-up pass before the first measured run.
- Measure p95 TTFT and TPOT, and goodput under the SLO. Record the rate, the seed and the version of each tool.
- Find the knee: the rate where p95 rises steeply and goodput stops tracking the offered rate. Plan capacity below it, and check the result with a second tool before you trust it.

## Try it
Little's law check (lesson 26): for each spread row, the average number of requests in the server is the rate times the mean end-to-end time in seconds. The count includes the requests waiting in the queue. Compute it for each rate in `RESULTS`. Which is the first rate where it goes above the 4 slots (`--max-num-seqs 4`)?

In [21]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [22]:
for k in sorted(k for k in RESULTS if k.startswith("spread")):
    r = RESULTS[k]; w = r["mean_e2el_ms"] / 1000
    print(f"{k}: {r['rate']:.2f}/s x {w:.1f} s = {r['rate'] * w:.1f} requests in the server (slots: 4)")
over = [k for k in sorted(RESULTS) if k.startswith("spread") and RESULTS[k]["rate"] * RESULTS[k]["mean_e2el_ms"] / 1000 > 4]
print("first rate above the 4 slots:", over[0] if over else "none")

spread_0.05: 0.05/s x 12.1 s = 0.6 requests in the server (slots: 4)
spread_0.10: 0.10/s x 17.5 s = 1.8 requests in the server (slots: 4)
spread_0.15: 0.15/s x 13.8 s = 2.1 requests in the server (slots: 4)
spread_0.40: 0.40/s x 25.5 s = 10.2 requests in the server (slots: 4)
first rate above the 4 slots: spread_0.40


## Recap
- An open-loop test sends requests on random gaps, as users do; a closed-loop test waits for each answer and hides the queue.
- Lengths should vary like real traffic. Spreading is theory until hundreds of requests show it; at 16 requests the fixed and spread runs look alike.
- Judge a run by p95 TTFT, p95 TPOT and goodput under an SLO, not by a count, an average or raw tokens per second.
- Bracket the knee of the sweep (between 0.15 and 0.40 requests per second in the recorded run), and plan capacity below it. Repeat the check with a second tool before you trust the number.
- **Interview:** "How would you load-test an LLM endpoint?" Strong answer: open-loop Poisson arrivals with a spread of realistic lengths, a warm-up pass, a rate sweep to the knee, and p95 TTFT, TPOT and goodput under the SLO, on the target hardware.